In [6]:
# -----######-----###### MONO & TRUE PEAK NORMALIZE MP4 AUDIO IN FOLDER -----######-----###### #
import subprocess
from pathlib import Path
from tqdm import tqdm

def _video_0108_vmono_peaknorm_GET_cleanfolder(folder_path):
    folder = Path(folder_path)
    if not folder.exists():
        print("❌ Folder not found.")
        return

    video_paths = [p for p in folder.glob("*.mp4") if not p.name.startswith('._')]

    for vid_path in tqdm(video_paths, desc="🎬 Normalizing Videos"):
        temp_mono_norm = vid_path.with_name("temp_audio_fixed.wav")
        temp_final = vid_path.with_name(f"temp_{vid_path.name}")

        # 1. Extract and convert audio to mono WAV, normalize to 0.0 dBFS peak using volume filter
        cmd1 = [
            "ffmpeg", "-i", str(vid_path),
            "-vn", "-ac", "1", "-ar", "44100",  # mono, 44.1kHz
            "-filter:a", "volumedetect", "-f", "null", "-"
        ]

        # Run volumedetect to measure current peak
        result = subprocess.run(cmd1, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)
        lines = result.stderr.split('\n')
        peak_db = None
        for line in lines:
            if "max_volume:" in line:
                try:
                    peak_db = float(line.split("max_volume:")[1].strip().replace(" dB", ""))
                except:
                    pass

        if peak_db is None:
            print(f"⚠️ Skipped {vid_path.name}: couldn't detect peak.")
            continue

        gain_db = -peak_db  # amount needed to reach 0 dBFS
        gain_db = round(gain_db, 2)

        # 2. Apply gain to normalize
        cmd2 = [
            "ffmpeg", "-i", str(vid_path),
            "-vn", "-ac", "1", "-ar", "44100",
            "-af", f"volume={gain_db}dB",
            "-y", str(temp_mono_norm)
        ]

        # 3. Replace video audio
        cmd3 = [
            "ffmpeg", "-i", str(vid_path), "-i", str(temp_mono_norm),
            "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
            "-y", str(temp_final)
        ]

        try:
            subprocess.run(cmd2, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            subprocess.run(cmd3, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
            vid_path.unlink()
            temp_final.rename(vid_path)
        except subprocess.CalledProcessError as e:
            print(f"⚠️ Error with {vid_path.name}: {e}")
        finally:
            if temp_mono_norm.exists():
                temp_mono_norm.unlink()


In [7]:
# Example folder path
folder_path = "/Users/yerik/Desktop/_RECAP_posts copy"

_video_0108_vmono_peaknorm_GET_cleanfolder(folder_path)



🎬 Normalizing Videos: 100%|██████████████████████████████████████████████████| 27/27 [00:20<00:00,  1.29it/s]
